# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and `@id`s relevant for referencing records in later analysis.

In [ ]:
# List all record sets with their @id, name, and description
record_sets = metadata.record_set or []  # List of RecordSets
if not record_sets:
    raise RuntimeError('This dataset does not expose record sets in the Croissant top-level, but you must use @id for reference.')
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '') if rs.get('description') else ''}")
    print('  Fields:')
    for field in rs['field']:
        print(f"    Field @id: {field['@id']} | Name: {field.get('name', '')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# --- Customize with your RecordSet @ids ---
# Based on inspection above; if record sets are not top-level, you may need to consult the Croissant JSON file for these ids.
# Here we adapt for the common case where the main tabular data is a single record set.

# Example: Suppose main record set is '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/rs0'
main_record_set_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/rs0'

record_sets_ids = [main_record_set_id]
dataframes = {}

for record_set in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set} with {len(records)} records.")
        print("Columns:", dataframes[record_set].columns.tolist())
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

# Preview the main DataFrame
df = dataframes.get(main_record_set_id)
if df is not None:
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming distributions, or grouping data by key attributes.

In [ ]:
# Example usage, replace these IDs with ones specific to your RecordSet/fields:
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/rs0/field_MW'
group_field_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/rs0/field_accession'

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Filtering for proteins with MW > 10 kDa
    threshold = 10_000  # e.g. 10,000 Da
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with MW (Da) > {threshold}: {len(filtered_df)} records")

    # Normalize the MW column
    normalized_col = numeric_field_id + '_normalized'
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by accession (if grouping field exists), aggregate mean MW
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename({numeric_field_id: 'mean_MW'}, axis=1)
        print(f"Mean MW per accession:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of molecular weights (MW) and the relationship between MW and other variables.

In [ ]:
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(10,6))
    df[numeric_field_id].hist(bins=50, color='skyblue', edgecolor='k')
    plt.xlabel('Molecular Weight (Da)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Molecular Weight (MW)')
    plt.show()

    # Scatter plot: MW vs number of peptides if field exists
    peptide_count_id = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2/rs0/field_peptide_count'
    if peptide_count_id in df.columns:
        plt.figure(figsize=(8,6))
        plt.scatter(df[numeric_field_id], df[peptide_count_id], alpha=0.5)
        plt.xlabel('Molecular Weight (Da)')
        plt.ylabel('Peptide Count')
        plt.title('MW vs Peptide Count')
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> mass spectrometry dataset, surveyed its schema through the Croissant framework, and demonstrated programmatic data exploration using `mlcroissant`.

**Main findings:**
- The dataset provides protein-level quantification and attributes, including molecular weight, accessions, and peptide counts, suitable for further exploratory and statistical analysis.
- The schema's explicit `@id` identifiers make it easy to reference and automate field access reliably.
- You can now adapt the workflow above for additional filtering, normalizing, and visualization, or carry forward to modeling tasks.